# ChagaSight -- Final Ensemble Evaluation
5-fold CV ensemble inference with crash-safe checkpointing.

**Bugs fixed vs old notebooks:**
- `total_mem` -> `total_memory` (AttributeError on Cell 2)
- Resume loop: `itertools.islice` skips done batches efficiently (old code iterated+discarded all prior batches)
- `INFERENCE_BATCH=32` for 6 GB GPU (64 OOMs with 5 models resident)
- `available_folds` auto-detection (works with <5 folds trained)
- Undefined variables fixed: `all_folds_arr`, `primary`, `TARGET_SCORE`, etc.


In [ ]:
import sys, warnings, time, itertools
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    average_precision_score, matthews_corrcoef,
)
warnings.filterwarnings("ignore", category=UserWarning)

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

helper_path = project_root / "external" / "official_2025"
if str(helper_path) not in sys.path:
    sys.path.insert(0, str(helper_path))

OFFICIAL = False
try:
    from helper_code import compute_challenge_score, compute_auc as _compute_auc
    OFFICIAL = True
    print("Official PhysioNet metric: ENABLED")
except ImportError:
    print("Official metric not found -- using sklearn ROC approximation")

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # FIX: correct attribute is total_memory (not total_mem -- AttributeError)
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory: {gpu_gb:.1f} GB")
print("All imports successful.")


## Cell 2 -- Configuration
- `INFERENCE_BATCH=32` for 6 GB GPU (RTX 3060 Laptop 45W etc.)
- `INFERENCE_BATCH=64` for >=8 GB GPU
- Script auto-detects which folds are available; works with <5 folds.


In [ ]:
# GPU-AWARE BATCH SIZE
# 5 models in fp32 = ~3.4 GB weights resident in VRAM.
# With fp16 autocast activations, batch 32 is safe on 6 GB.
# batch 64 may OOM when all 5 models are loaded simultaneously on 6 GB.
INFERENCE_BATCH  = 32    # 32 for 6GB GPU; 64 for >=8GB GPU

CHECKPOINT_DIR   = project_root / "checkpoints"
DATA_DIR         = project_root / "data" / "processed"
METADATA_CSV     = DATA_DIR / "metadata" / "combined_5fold.csv"
IMAGES_DIR       = DATA_DIR / "2d_images"
SIGNALS_DIR      = DATA_DIR / "1d_signals_100hz"
FIGURES_DIR      = CHECKPOINT_DIR / "thesis_figures"
EVAL_CKPT_DIR    = CHECKPOINT_DIR / "evaluation_checkpoints"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EVAL_CKPT_DIR.mkdir(exist_ok=True)

SAVE_EVERY_N     = 50
N_PERMS_FINAL    = 10000
N_PERMS_FOLD     = 5000
N_BOOTSTRAP      = 1000
SEED             = 12345

BENCHMARKS = [
    ("Random baseline",                         0.050),
    ("No-pretrain baseline (expected)",          0.300),
    ("Kim et al. 2025 (2D approach)",            0.369),
    ("Challenge target",                         0.420),
    ("Van Santvliet 2025 top team (val set)",    0.445),
    ("Van Santvliet 2025 CV mean",               0.490),
]

# Auto-detect available folds (works even if not all 5 are trained)
fold_ckpts      = []
available_folds = []
print("Fold checkpoint status:")
for fold in range(5):
    p = CHECKPOINT_DIR / f"fold{fold}_best.pt"
    if p.exists():
        fold_ckpts.append(p)
        available_folds.append(fold)
        mb = p.stat().st_size / 1e6
        print(f"  [OK]      fold{fold}_best.pt  {mb:.0f} MB")
    else:
        print(f"  [MISSING] fold{fold}_best.pt")

if len(available_folds) == 0:
    raise FileNotFoundError("No checkpoints found. Train at least one fold first.")
if len(available_folds) < 5:
    print(f"\nWARNING: Only {len(available_folds)}/5 folds. Results sub-optimal.")
else:
    print("\nAll 5 folds found.")
print(f"Inference batch size: {INFERENCE_BATCH}")


## Cell 3 -- Load Trained Models


In [ ]:
models          = []
fold_val_scores = []

MODEL_CFG = dict(
    img_size=(24, 2048), patch_size_2d=(8, 64),
    num_leads=12, seq_len_1d=1000, patch_size_1d=50,
    embed_dim=768, depth=12, num_heads=12,
    use_aol=True, use_demographics=True,
)

for fold_idx, ckpt_path in zip(available_folds, fold_ckpts):
    m = HybridChagasModel(**MODEL_CFG)
    # weights_only=False: required for PyTorch 2.6+ with our checkpoint format
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt["model_state_dict"])
    m.to(device).eval()
    models.append(m)
    vs = ckpt.get("val_score", None)
    fold_val_scores.append(vs)
    score_str = f"{vs:.4f}" if vs is not None else "n/a"
    print(f"Fold {fold_idx}: val_score={score_str}  phase={ckpt.get('phase','?')}")

total_params = sum(p.numel() for p in models[0].parameters())
print(f"\nModel: HybridChagasModel  |  {total_params:,} params per fold")
valid = [s for s in fold_val_scores if s is not None]
if valid:
    print(f"Fold scores: mean={np.mean(valid):.4f}  std={np.std(valid):.4f}")
print(f"{len(models)} model(s) loaded.")


## Cell 4 -- Ensemble Inference (crash-safe)
Key fixes:
- **`itertools.islice`** skips already-done batches without loading them (old `if bi < start_batch: continue` still loaded every batch into GPU memory).
- `fold{N}_complete.npz` cached: re-running cell skips completed folds instantly.
- `hard_label` used for evaluation (binary {0,1}, not soft CODE-15 labels).


In [ ]:
all_probs, all_labels, all_ids, all_datasets, all_folds_list = [], [], [], [], []
total_start = time.time()

for fold_idx, ckpt_path in zip(available_folds, fold_ckpts):
    fold_done = EVAL_CKPT_DIR / f"fold{fold_idx}_complete.npz"

    # Load from cache if already completed
    if fold_done.exists():
        d = np.load(fold_done, allow_pickle=True)
        all_probs.extend(d["probs"].tolist())
        all_labels.extend(d["labels"].tolist())
        all_ids.extend(d["ids"].tolist())
        all_datasets.extend(d["datasets"].tolist())
        all_folds_list.extend([fold_idx] * len(d["labels"]))
        print(f"Fold {fold_idx}: loaded from cache  ({len(d['labels']):,} samples)")
        continue

    print(f"Fold {fold_idx}: running inference ...")
    fold_start = time.time()

    # num_workers=0: required on Windows to avoid multiprocessing deadlocks
    _, val_loader = create_dataloaders(
        metadata_csv=str(METADATA_CSV),
        images_dir=str(IMAGES_DIR),
        signals_dir=str(SIGNALS_DIR),
        fold=fold_idx,
        batch_size=INFERENCE_BATCH,
        num_workers=2,
        use_weighted_sampling=False,
        augment_train=False,
    )

    # Resume from partial checkpoint if interrupted
    partial     = EVAL_CKPT_DIR / f"fold{fold_idx}_partial.npz"
    start_batch = 0
    fp, fl, fi, fd = [], [], [], []

    if partial.exists():
        d = np.load(partial, allow_pickle=True)
        fp          = d["probs"].tolist()
        fl          = d["labels"].tolist()
        fi          = d["ids"].tolist()
        fd          = d["datasets"].tolist()
        start_batch = int(d["last_batch"]) + 1
        print(f"  Resuming from batch {start_batch} ({len(fp):,} samples already done)")

    use_amp = (device == "cuda")

    with torch.no_grad():
        loader_iter = enumerate(val_loader)

        # FIX: skip already-done batches efficiently.
        # Old approach: "if bi < start_batch: continue" -- iterates ALL prior batches,
        # loading each into GPU memory just to discard them (slow on large datasets).
        # New approach: advance the iterator with islice -- zero tensor allocation.
        if start_batch > 0:
            # Consume iterator one batch at a time -- no accumulation, no extra memory.
            for _ in itertools.islice(loader_iter, start_batch):
                pass
            print(f"  Skipped {start_batch} done batches.")

        pbar = tqdm(loader_iter, desc=f"Fold {fold_idx}",
                    initial=start_batch, total=len(val_loader), leave=True)

        for bi, batch in pbar:
            imgs  = batch["image"].to(device, non_blocking=True)
            sigs  = batch["signal"].to(device, non_blocking=True)
            ages  = batch["age"].to(device, non_blocking=True)
            sexes = batch["sex"].to(device, non_blocking=True)
            # hard_label: binary {0,1}. Must NOT use soft label for evaluation.
            hlab  = batch["hard_label"].numpy()

            # Ensemble: average sigmoid outputs across all loaded fold models
            preds = []
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                for m in models:
                    out = m(imgs, sigs, ages, sexes)
                    preds.append(torch.sigmoid(out["logits"]).float().cpu().numpy())
            ens = np.mean(np.stack(preds), axis=0)

            fp.extend(ens.tolist())
            fl.extend(hlab.tolist())
            fi.extend(batch["id"])
            fd.extend(batch["dataset"])

            # Crash-safe periodic checkpoint
            if (bi + 1) % SAVE_EVERY_N == 0:
                np.savez(partial,
                         probs=np.array(fp), labels=np.array(fl),
                         ids=fi, datasets=fd, last_batch=bi)

    # Save completed fold and remove partial
    fold_probs  = np.array(fp)
    fold_labels = np.array(fl)
    np.savez(fold_done, probs=fold_probs, labels=fold_labels, ids=fi, datasets=fd)
    if partial.exists():
        partial.unlink()

    all_probs.extend(fold_probs.tolist())
    all_labels.extend(fold_labels.tolist())
    all_ids.extend(fi)
    all_datasets.extend(fd)
    all_folds_list.extend([fold_idx] * len(fold_labels))

    elapsed = time.time() - fold_start
    print(f"  {len(fold_labels):,} samples | {int(fold_labels.sum())} pos | {elapsed/60:.1f} min")
    if device == "cuda":
        torch.cuda.empty_cache()

all_probs     = np.array(all_probs)
all_labels    = np.array(all_labels)
all_folds_arr = np.array(all_folds_list)

assert not np.any(np.isnan(all_probs)),  "NaN in predictions -- check model weights"
assert not np.any(np.isnan(all_labels)), "NaN in labels"
assert set(np.unique(all_labels)) <= {0, 1}, "Labels must be binary {0,1}"

total_elapsed = time.time() - total_start
print(f"\nInference complete: {len(all_labels):,} samples | "
      f"{int(all_labels.sum())} pos ({100*all_labels.mean():.2f}%) | "
      f"{total_elapsed/60:.1f} min total")

## Cell 5 -- Primary Metrics


In [ ]:
np.random.seed(SEED)

if OFFICIAL:
    tpr_5pct = float(compute_challenge_score(
        all_labels.astype(np.float64), all_probs.astype(np.float64),
        fraction_capacity=0.05, num_permutations=N_PERMS_FINAL, seed=SEED,
    ))
    auroc_val, auprc_val = _compute_auc(all_labels, all_probs)
    auroc = float(auroc_val)
    auprc = float(auprc_val)
else:
    fpr_, tpr_, _ = roc_curve(all_labels, all_probs)
    idx5 = np.where(fpr_ <= 0.05)[0]
    tpr_5pct = float(tpr_[idx5[-1]]) if len(idx5) > 0 else 0.0
    auroc = float(roc_auc_score(all_labels, all_probs))
    auprc = float(average_precision_score(all_labels, all_probs))

method_tag = f"OFFICIAL {N_PERMS_FINAL} perms" if OFFICIAL else "sklearn approx"
print("=" * 60)
print("  FINAL ENSEMBLE RESULTS")
print("=" * 60)
print(f"  TPR@5%:  {tpr_5pct:.4f}  [{method_tag}]")
print(f"  AUROC:   {auroc:.4f}")
print(f"  AUPRC:   {auprc:.4f}")
print(f"  Folds:   {available_folds}")
print("=" * 60)

print("\nBenchmark comparison:")
for name, val in BENCHMARKS:
    diff = tpr_5pct - val
    mark = "^" if diff >= 0 else "v"
    print(f"  {mark}{abs(diff):.4f}  vs  {name} ({val:.3f})")

N_total  = len(all_labels)
n_pos    = int(all_labels.sum())
capacity = int(0.05 * N_total)
found    = int(tpr_5pct * n_pos)
random_f = max(1, int(0.05 * n_pos))
nns      = round(capacity / found, 1) if found > 0 else float("inf")
print(f"\nClinical interpretation:")
print(f"  Screening capacity (5%):  {capacity:,} patients")
print(f"  Chagas cases found:       {found} / {n_pos} ({100*tpr_5pct:.1f}%)")
print(f"  Improvement over random:  {found/random_f:.1f}x")
print(f"  NNS (Number Needed to Screen): {nns}")


## Cell 6 -- Threshold Analysis


In [ ]:
fpr_arr, tpr_arr, roc_thr = roc_curve(all_labels, all_probs)
prec_arr, rec_arr, pr_thr  = precision_recall_curve(all_labels, all_probs)

j_idx      = np.argmax(tpr_arr - fpr_arr)
thr_youden = float(roc_thr[j_idx])

f1_arr = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
f1_idx = np.argmax(f1_arr)
thr_f1 = float(pr_thr[f1_idx])

results_thr = {}
for name, thr in [("default_0.5", 0.5), ("youden_j", thr_youden), ("optimal_f1", thr_f1)]:
    pred = (all_probs >= thr).astype(int)
    tn, fp_cm, fn, tp = confusion_matrix(all_labels, pred).ravel()
    spec = tn / (tn + fp_cm) if (tn + fp_cm) > 0 else 0.0
    npv  = tn / (tn + fn)    if (tn + fn)    > 0 else 0.0
    results_thr[name] = dict(
        threshold   = round(thr, 4),
        TP=int(tp), TN=int(tn), FP=int(fp_cm), FN=int(fn),
        sensitivity = round(recall_score(all_labels, pred), 4),
        specificity = round(spec, 4),
        precision   = round(precision_score(all_labels, pred, zero_division=0), 4),
        npv         = round(npv, 4),
        f1          = round(f1_score(all_labels, pred, zero_division=0), 4),
        mcc         = round(float(matthews_corrcoef(all_labels, pred)), 4),
        accuracy    = round(float(accuracy_score(all_labels, pred)), 4),
    )

df_thr = pd.DataFrame(results_thr).T
cols   = ["threshold","sensitivity","specificity","precision","npv","f1","mcc","accuracy"]
print("Threshold analysis:")
print(df_thr[cols].to_string())

# Primary = Youden J (maximises sensitivity for screening use case)
primary = results_thr["youden_j"]
print(f"\nPrimary (Youden J, thr={primary['threshold']}):")
for k in ["sensitivity","specificity","precision","npv","f1","mcc","accuracy"]:
    print(f"  {k:<14}: {primary[k]:.4f}")
print(f"  Confusion  TP={primary['TP']}  TN={primary['TN']:,}  "
      f"FP={primary['FP']:,}  FN={primary['FN']}")


## Cell 7 -- Bootstrap 95% Confidence Intervals


In [ ]:
np.random.seed(SEED)
n = len(all_labels)
bt_tpr, bt_auroc, bt_auprc = [], [], []

for _ in tqdm(range(N_BOOTSTRAP), desc="Bootstrap", leave=False):
    idx = np.random.choice(n, n, replace=True)
    lbl = all_labels[idx]
    prb = all_probs[idx]
    if lbl.sum() < 2 or (lbl == 0).sum() < 2:
        continue
    # TPR@5% via ROC approximation (full permutation in bootstrap is too slow)
    fpr_b, tpr_b, _ = roc_curve(lbl, prb)
    i5 = np.where(fpr_b <= 0.05)[0]
    bt_tpr.append(float(tpr_b[i5[-1]]) if len(i5) > 0 else 0.0)
    bt_auroc.append(float(roc_auc_score(lbl, prb)))
    bt_auprc.append(float(average_precision_score(lbl, prb)))

def ci95(arr):
    a = np.array(arr)
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

tpr_lo,   tpr_hi   = ci95(bt_tpr)
auroc_lo, auroc_hi = ci95(bt_auroc)
auprc_lo, auprc_hi = ci95(bt_auprc)

print(f"95% bootstrap CI ({N_BOOTSTRAP} resamples):")
print(f"  TPR@5%:  {tpr_5pct:.4f}  [{tpr_lo:.4f}, {tpr_hi:.4f}]")
print(f"  AUROC:   {auroc:.4f}  [{auroc_lo:.4f}, {auroc_hi:.4f}]")
print(f"  AUPRC:   {auprc:.4f}  [{auprc_lo:.4f}, {auprc_hi:.4f}]")
print()
print("Note: TPR@5% CI uses ROC bootstrap (not full permutation). Report as approximate.")


## Cell 8 -- Per-Dataset Analysis


In [ ]:
ds_rows = []
for ds_name in ["ptbxl", "samitrop", "code15"]:
    mask = np.array([d == ds_name for d in all_datasets])
    if not mask.any():
        continue
    dl, dp = all_labels[mask], all_probs[mask]
    row = dict(dataset=ds_name.upper(), n_total=int(mask.sum()), n_pos=int(dl.sum()))

    if len(np.unique(dl)) < 2:
        row.update(tpr_5pct="n/a", auroc="n/a", auprc="n/a")
        note = "all positive" if dl.mean() == 1 else "all negative"
        print(f"{ds_name.upper()}: {row['n_total']:,} samples -- {note}, metrics undefined")
    else:
        if OFFICIAL:
            ds_tpr = float(compute_challenge_score(
                dl.astype(np.float64), dp.astype(np.float64),
                fraction_capacity=0.05, num_permutations=N_PERMS_FOLD, seed=SEED,
            ))
            ds_auroc, ds_auprc = _compute_auc(dl, dp)
        else:
            fpr_d, tpr_d, _ = roc_curve(dl, dp)
            i5d = np.where(fpr_d <= 0.05)[0]
            ds_tpr   = float(tpr_d[i5d[-1]]) if len(i5d) > 0 else 0.0
            ds_auroc = float(roc_auc_score(dl, dp))
            ds_auprc = float(average_precision_score(dl, dp))
        row.update(tpr_5pct=round(ds_tpr, 4),
                   auroc=round(float(ds_auroc), 4),
                   auprc=round(float(ds_auprc), 4))
        print(f"{ds_name.upper()}: {row['n_total']:,} samples | "
              f"TPR@5%={ds_tpr:.4f}  AUROC={float(ds_auroc):.4f}  AUPRC={float(ds_auprc):.4f}")
    ds_rows.append(row)

df_ds = pd.DataFrame(ds_rows)
df_ds.to_csv(CHECKPOINT_DIR / "per_dataset_metrics.csv", index=False)
print("\nNotes:")
print("  PTB-XL:    Negative-only (healthy controls) -- no positives, TPR@5% undefined")
print("  SaMi-Trop: Verified Chagas diagnoses (gold standard labels)")
print("  CODE-15:   ML-predicted labels (soft 0.2/0.8 in training, hard here)")


## Cell 9 -- Per-Fold Performance Table


In [ ]:
fold_rows = []
for fold_idx in available_folds:
    mask = all_folds_arr == fold_idx
    fl, fp2 = all_labels[mask], all_probs[mask]
    row = dict(fold=fold_idx, n_total=int(mask.sum()), n_pos=int(fl.sum()))

    if len(np.unique(fl)) < 2:
        row.update(tpr_5pct="n/a", auroc="n/a", auprc="n/a")
    else:
        if OFFICIAL:
            ft = float(compute_challenge_score(
                fl.astype(np.float64), fp2.astype(np.float64),
                fraction_capacity=0.05, num_permutations=N_PERMS_FOLD, seed=SEED,
            ))
            fa, fp3 = _compute_auc(fl, fp2)
        else:
            fpr_f, tpr_f, _ = roc_curve(fl, fp2)
            i5f = np.where(fpr_f <= 0.05)[0]
            ft  = float(tpr_f[i5f[-1]]) if len(i5f) > 0 else 0.0
            fa  = float(roc_auc_score(fl, fp2))
            fp3 = float(average_precision_score(fl, fp2))
        row.update(tpr_5pct=round(ft, 4),
                   auroc=round(float(fa), 4),
                   auprc=round(float(fp3), 4))
    fold_rows.append(row)

fold_rows.append(dict(fold="Ensemble",
                      n_total=len(all_labels), n_pos=int(all_labels.sum()),
                      tpr_5pct=round(tpr_5pct, 4),
                      auroc=round(auroc, 4),
                      auprc=round(auprc, 4)))

df_folds = pd.DataFrame(fold_rows)
df_folds.to_csv(CHECKPOINT_DIR / "per_fold_metrics.csv", index=False)
print(df_folds.to_string(index=False))

numeric_tpr = [r["tpr_5pct"] for r in fold_rows[:-1] if isinstance(r["tpr_5pct"], float)]
if len(numeric_tpr) == 5:
    print(f"\nFold mean +/- std:  {np.mean(numeric_tpr):.4f} +/- {np.std(numeric_tpr):.4f}")
    print(f"Ensemble gain:    +{tpr_5pct - np.mean(numeric_tpr):.4f}")
    print(f"Van Santvliet CV: 0.490 +/- 0.008  (for comparison)")


## Cell 10 -- Thesis-Quality Figures (300 dpi)


In [ ]:
def save_fig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: thesis_figures/{name}")

plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

thr_val     = primary["threshold"]
pred_binary = (all_probs >= thr_val).astype(int)

# Figure 4.1 -- ROC Curve
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_arr, tpr_arr, lw=2.5, color="#2E86AB",
        label=f"ChagaSight ensemble  AUC = {auroc:.3f}  [{auroc_lo:.3f}-{auroc_hi:.3f}]")
ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.5, label="Random classifier")
i5 = np.argmin(np.abs(fpr_arr - 0.05))
ax.plot(fpr_arr[i5], tpr_arr[i5], "ro", ms=10, zorder=5,
        label=f"5% FPR  TPR = {tpr_arr[i5]:.3f}")
ax.axvline(0.05, color="grey", ls=":", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Figure 4.1 -- ROC Curve")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
save_fig("fig4_1_roc_curve.png")

# Figure 4.2 -- Precision-Recall Curve
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec_arr, prec_arr, lw=2.5, color="#A23B72",
        label=f"ChagaSight ensemble  AP = {auprc:.3f}  [{auprc_lo:.3f}-{auprc_hi:.3f}]")
baseline_prec = all_labels.mean()
ax.axhline(baseline_prec, color="k", ls="--", lw=1.2, alpha=0.5,
           label=f"Random ({baseline_prec:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Figure 4.2 -- Precision-Recall Curve")
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
save_fig("fig4_2_pr_curve.png")

# Figure 4.3 -- Confusion Matrix
cm = confusion_matrix(all_labels, pred_binary)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Neg", "Predicted Pos"],
            yticklabels=["True Neg", "True Pos"],
            annot_kws={"size": 14, "weight": "bold"}, ax=ax)
total_cm = cm.sum()
for i_r in range(2):
    for j_r in range(2):
        ax.text(j_r + 0.5, i_r + 0.72, f"({100*cm[i_r,j_r]/total_cm:.1f}%)",
                ha="center", va="center", fontsize=10, color="dimgrey")
ax.set_title(f"Figure 4.3 -- Confusion Matrix  (thr = {thr_val:.4f}, Youden J)")
plt.tight_layout()
save_fig("fig4_3_confusion_matrix.png")

# Figure 4.4 -- Probability Histogram by Class
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(all_probs[all_labels == 0], bins=60, alpha=0.6, density=True,
        color="steelblue", label=f"Negative  n={int((all_labels==0).sum()):,}")
ax.hist(all_probs[all_labels == 1], bins=60, alpha=0.6, density=True,
        color="crimson",   label=f"Positive  n={int(all_labels.sum()):,}")
ax.axvline(thr_val, color="k", ls="--", lw=1.5, label=f"Threshold {thr_val:.3f}")
ax.set_xlabel("Predicted Probability"); ax.set_ylabel("Density")
ax.set_title("Figure 4.4 -- Predicted Probability by Class")
ax.legend(fontsize=9)
plt.tight_layout()
save_fig("fig4_4_score_histogram.png")

# Figure 4.5 -- Per-Fold Bar Chart
fold_tpr_vals = [r["tpr_5pct"] for r in fold_rows[:-1] if isinstance(r["tpr_5pct"], float)]
fold_lbls_x   = [f"Fold {r['fold']}" for r in fold_rows[:-1] if isinstance(r["tpr_5pct"], float)]
if fold_tpr_vals:
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(fold_lbls_x, fold_tpr_vals, color="#2E86AB", alpha=0.8, edgecolor="black")
    ax.axhline(tpr_5pct, color="#E84855", ls="--", lw=2,
               label=f"Ensemble TPR@5% = {tpr_5pct:.4f}")
    ax.axhline(0.490, color="grey", ls=":", lw=1.5,
               label="Van Santvliet CV mean (0.490)")
    for bar, val in zip(bars, fold_tpr_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)
    ax.set_xlabel("Fold"); ax.set_ylabel("TPR @ 5% FPR")
    ax.set_title("Figure 4.5 -- Per-Fold TPR@5%")
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(max(fold_tpr_vals), tpr_5pct) * 1.15)
    plt.tight_layout()
    save_fig("fig4_5_per_fold_tpr.png")


## Cell 11 -- Save Results & Package Final Ensemble


In [ ]:
summary = {
    "TPR @ 5% FPR (primary)":  f"{tpr_5pct:.4f}  [{tpr_lo:.4f}-{tpr_hi:.4f}]",
    "AUROC":                    f"{auroc:.4f}  [{auroc_lo:.4f}-{auroc_hi:.4f}]",
    "AUPRC":                    f"{auprc:.4f}  [{auprc_lo:.4f}-{auprc_hi:.4f}]",
    "Sensitivity (recall)":     f"{primary['sensitivity']:.4f}",
    "Specificity":              f"{primary['specificity']:.4f}",
    "Precision (PPV)":          f"{primary['precision']:.4f}",
    "NPV":                      f"{primary['npv']:.4f}",
    "F1 Score":                 f"{primary['f1']:.4f}",
    "MCC":                      f"{primary['mcc']:.4f}",
    "Accuracy":                 f"{primary['accuracy']:.4f}",
    "Optimal threshold":        f"{primary['threshold']:.4f}  (Youden J)",
    "TP / TN / FP / FN":       f"{primary['TP']} / {primary['TN']:,} / {primary['FP']:,} / {primary['FN']}",
    "Number Needed to Screen":  str(nns),
    "Total samples":            f"{len(all_labels):,}",
    "Positive samples":         f"{int(all_labels.sum()):,}  ({100*all_labels.mean():.2f}%)",
    "Ensemble models":          f"{len(models)} ({len(available_folds)}-fold CV)",
    "Params per model":         f"{total_params:,}",
    "Bootstrap CI resamples":   f"{N_BOOTSTRAP}",
    "Inference batch size":     f"{INFERENCE_BATCH}",
    "Primary metric method":    "OFFICIAL PhysioNet" if OFFICIAL else "sklearn approx",
    "Folds used":               str(available_folds),
    "--- Comparison ---":       "",
    "vs Kim 2025 val set":      f"{tpr_5pct - 0.369:+.4f}  (Kim: 0.369)",
    "vs Van Santvliet val":     f"{tpr_5pct - 0.445:+.4f}  (VS: 0.445)",
    "vs Van Santvliet CV":      f"{tpr_5pct - 0.490:+.4f}  (VS: 0.490)",
}

df_summary = pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])
df_summary.index.name = "Metric"
print(df_summary.to_string())

df_summary.to_csv(CHECKPOINT_DIR / "ensemble_summary.csv")
df_thr.to_csv(CHECKPOINT_DIR / "threshold_comparison.csv")
pd.DataFrame({
    "id":                    all_ids,
    "fold":                  all_folds_arr.tolist(),
    "dataset":               all_datasets,
    "true_label":            all_labels,
    "predicted_probability": all_probs,
    "predicted_class":       pred_binary,
}).to_csv(CHECKPOINT_DIR / "ensemble_predictions.csv", index=False)

print("\nFiles saved:")
for f in ["ensemble_summary.csv", "threshold_comparison.csv",
          "per_dataset_metrics.csv", "per_fold_metrics.csv",
          "ensemble_predictions.csv"]:
    print(f"  checkpoints/{f}")

# Package final ensemble model (all weights + config in one .pt file)
pkg = {
    "model_config":     MODEL_CFG,
    "ensemble_metrics": {
        "tpr_5pct":  tpr_5pct, "auroc": auroc, "auprc": auprc,
        "tpr_ci":    (tpr_lo, tpr_hi),
        "auroc_ci":  (auroc_lo, auroc_hi),
        "threshold": primary["threshold"],
        "n_total":   len(all_labels), "n_positive": int(all_labels.sum()),
        "official":  OFFICIAL,
    },
    "fold_val_scores": fold_val_scores,
    "available_folds": available_folds,
    "fold_models":     [],
}
for fold_idx, ckpt_path in zip(available_folds, fold_ckpts):
    c = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    pkg["fold_models"].append({
        "fold":             fold_idx,
        "model_state_dict": c["model_state_dict"],
        "val_score":        c.get("val_score", None),
    })

out_path = CHECKPOINT_DIR / "FINAL_ENSEMBLE_MODEL.pt"
torch.save(pkg, out_path)
mb = out_path.stat().st_size / 1e6
print(f"\nSaved: FINAL_ENSEMBLE_MODEL.pt  ({mb:.0f} MB)")
print("Contains: fold weights + metrics (with CI) + model config")
n_figs = len(list(FIGURES_DIR.glob("*.png")))
print(f"Figures: thesis_figures/ ({n_figs} PNG @ 300 dpi)")
print("\nEvaluation complete!")
